## 🎯 Learning Objectives
* Design and implement custom tools for a LangChain agent.
* Integrate multiple custom tools into a single agent workflow.
* Configure a LangChain agent with conversational memory to maintain context.
* Evaluate an agent's ability to effectively use tools to resolve customer queries.


## Exercise: Build a Tool-Using Customer Support Agent

### Task Description

In this exercise, you will build a sophisticated customer support agent for a fictional smart home device company, "AgenticGadgets Inc.". This agent will be responsible for assisting customers with common inquiries by leveraging a set of specialized tools. Your goal is to create an agent that can intelligently route customer questions to the appropriate tool, retrieve information, and provide helpful responses, all while maintaining conversational context.

### Scenario: AgenticGadgets Inc. Customer Support

AgenticGadgets Inc. sells various smart home devices. Customers frequently contact support for:
1.  **Product Information**: Inquiries about product specifications, features, or compatibility.
2.  **Order Status**: Questions regarding the current status of their purchases.
3.  **Troubleshooting**: Basic guidance for common issues with their devices.

### Requirements

1.  **LangChain Agent**: Implement the customer support agent using the LangChain framework.
2.  **Custom Tools**: Create at least **three (3)** custom tools that the agent can use:
    *   `ProductInfoTool`: Takes a product name (e.g., "SmartBulb Pro", "SmartLock Elite") and returns its specifications or features. Mock this with a dictionary lookup.
    *   `OrderStatusTool`: Takes an order ID (e.g., "AG12345", "AG67890") and returns its current status (e.g., "Shipped", "Processing", "Delivered"). Mock this with a simple function.
    *   `TroubleshootingGuideTool`: Takes a product name and a brief description of an issue, and returns a relevant troubleshooting step or link to a guide. Mock this with a dictionary lookup or simple conditional logic.
3.  **Conversational Memory**: The agent must maintain conversational memory to understand follow-up questions and provide context-aware responses.
4.  **Robustness**: The agent should gracefully handle cases where:
    *   A tool cannot find information (e.g., invalid product name, non-existent order ID).
    *   The query does not require a tool, and the agent can answer directly or ask for clarification.
    *   The query requires multiple steps or tools.
5.  **Modern LLM**: Utilize a modern, capable LLM (e.g., `gpt-4o`, `claude-3-5-sonnet`, `gemini-1.5-pro`) for the agent's reasoning.

### Evaluation Criteria

Your solution will be evaluated based on:
*   **Correctness**: Does the agent correctly use the tools to answer questions?
*   **Tool Usage**: Is the agent able to select the appropriate tool(s) for various queries?
*   **Memory Effectiveness**: Does the agent remember previous turns and use context effectively?
*   **Robustness**: How well does the agent handle edge cases and unexpected inputs?
*   **Code Quality**: Readability, comments, and adherence to best practices.
*   **Demonstration**: Provide examples of the agent handling different types of customer queries, including multi-turn conversations.


In [ ]:
# Ensure you have the necessary libraries installed:
# pip install langchain langchain-openai beautifulsoup4

import os
from typing import Dict, Any

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_react_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain.memory import ConversationBufferMemory

# --- Environment Setup ---
# Set your API key. Replace 'YOUR_OPENAI_API_KEY' with your actual key.
# It's recommended to load this from environment variables for production.
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

# Mock Data for Tools
PRODUCT_DATABASE = {
    "SmartBulb Pro": {
        "specs": "1000 lumens, E27 base, RGBW, Wi-Fi 6, Matter compatible",
        "features": "Voice control (Alexa/Google), scheduling, dimmable, color temperature adjustment."
    },
    "SmartLock Elite": {
        "specs": "Bluetooth 5.2, Wi-Fi 2.4GHz, AES-256 encryption, IP65 water resistance",
        "features": "Fingerprint unlock, keypad, remote access, auto-lock, tamper alarm."
    },
    "SmartThermostat X": {
        "specs": "7-inch LCD touchscreen, Z-Wave Plus, Energy Star certified, built-in humidity sensor",
        "features": "Learning algorithm, zone control, geofencing, energy usage reports."
    }
}

ORDER_DATABASE = {
    "AG12345": {"status": "Shipped", "tracking_id": "TRK987654", "estimated_delivery": "2026-03-10"},
    "AG67890": {"status": "Processing", "items": ["SmartBulb Pro", "SmartLock Elite"]},
    "AG11223": {"status": "Delivered", "delivery_date": "2026-02-28"},
    "AG44556": {"status": "Cancelled", "reason": "Customer request"}
}

TROUBLESHOOTING_GUIDES = {
    "SmartLock Elite": {
        "not connecting to wi-fi": "1. Ensure your Wi-Fi router is 2.4GHz. 2. Restart your router and the SmartLock. 3. Check for firmware updates in the app. 4. If issues persist, try a factory reset.",
        "battery draining fast": "1. Check for frequent unlocking/locking events. 2. Ensure firmware is up-to-date. 3. Reduce sensitivity of auto-lock feature. 4. Consider replacing with high-quality batteries."
    },
    "SmartBulb Pro": {
        "not responding": "1. Ensure the bulb is powered on. 2. Check Wi-Fi connection. 3. Try resetting the bulb by cycling power 5 times. 4. Re-pair with the app.",
        "flickering": "1. Ensure the bulb is compatible with your dimmer switch (if any). 2. Check for loose connections. 3. Try in a different fixture."
    }
}

# --- Define Custom Tools ---

@tool
def get_product_info(product_name: str) -> str:
    """Retrieves detailed information (specs, features) for a given product name.
    Use this tool when a user asks about product specifications, features, or details.
    Input should be the exact product name, e.g., 'SmartBulb Pro'.
    """
    product_name = product_name.strip()
    info = PRODUCT_DATABASE.get(product_name)
    if info:
        return f"Product: {product_name}\nSpecifications: {info['specs']}\nFeatures: {info['features']}"
    else:
        return f"Could not find information for product: {product_name}. Please check the product name."

@tool
def get_order_status(order_id: str) -> str:
    """Retrieves the current status and details for a given order ID.
    Use this tool when a user asks about their order's status, tracking, or delivery.
    Input should be the exact order ID, e.g., 'AG12345'.
    """
    order_id = order_id.strip()
    status_info = ORDER_DATABASE.get(order_id)
    if status_info:
        details = []
        for key, value in status_info.items():
            details.append(f"{key.replace('_', ' ').title()}: {value}")
        return f"Order ID: {order_id}\nStatus Details: {', '.join(details)}"
    else:
        return f"Could not find order with ID: {order_id}. Please verify the order ID."

@tool
def get_troubleshooting_guide(product_name: str, issue_description: str) -> str:
    """Provides troubleshooting steps for a specific product and issue.
    Use this tool when a user describes a problem with their device and needs help fixing it.
    Input should be the product name and a brief description of the issue, e.g., 'SmartLock Elite', 'not connecting to Wi-Fi'.
    """
    product_name = product_name.strip()
    issue_description = issue_description.strip().lower()

    product_issues = TROUBLESHOOTING_GUIDES.get(product_name)
    if product_issues:
        for issue_key, guide in product_issues.items():
            if issue_key in issue_description:
                return f"Troubleshooting steps for {product_name} - {issue_key}:\n{guide}"
        return f"No specific troubleshooting guide found for '{issue_description}' with {product_name}. Please describe the issue in more detail or try a different product."
    else:
        return f"No troubleshooting guides available for product: {product_name}."

# List of all tools available to the agent
agent_tools = [get_product_info, get_order_status, get_troubleshooting_guide]

# --- Initialize LLM ---
# Using gpt-4o for its advanced reasoning and function calling capabilities.
# Ensure OPENAI_API_KEY is set in your environment variables.
llm = ChatOpenAI(model="gpt-4o", temperature=0)

print("Setup complete. LLM and tools initialized.")


### Your Implementation

Now it's your turn! Based on the setup code provided above, implement the LangChain agent. Your solution should:

1.  **Define the Agent Prompt**: Create a `ChatPromptTemplate` that includes a system message, placeholders for chat history, and agent scratchpad.
2.  **Create the Agent**: Use `create_react_agent` (or `create_openai_tools_agent` if you prefer the OpenAI function calling approach) with your LLM, tools, and prompt.
3.  **Set up Memory**: Integrate `ConversationBufferMemory` to store and retrieve chat history.
4.  **Create Agent Executor**: Instantiate `AgentExecutor` with your agent, tools, and memory.
5.  **Test the Agent**: Write code to interact with your agent, demonstrating its ability to handle various customer queries, including:
    *   A product information query.
    *   An order status query.
    *   A troubleshooting query.
    *   A multi-turn conversation where context is maintained.
    *   A query that doesn't require a tool.
    *   A query for which a tool returns no results.

Feel free to add more mock data or refine the tools if you wish to expand the agent's capabilities.


In [ ]:
import os
from typing import Dict, Any

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_react_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain.memory import ConversationBufferMemory

# --- Environment Setup ---
# Set your API key. Replace 'YOUR_OPENAI_API_KEY' with your actual key.
# It's recommended to load this from environment variables for production.
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

# Mock Data for Tools (re-defined for self-contained solution cell)
PRODUCT_DATABASE = {
    "SmartBulb Pro": {
        "specs": "1000 lumens, E27 base, RGBW, Wi-Fi 6, Matter compatible",
        "features": "Voice control (Alexa/Google), scheduling, dimmable, color temperature adjustment."
    },
    "SmartLock Elite": {
        "specs": "Bluetooth 5.2, Wi-Fi 2.4GHz, AES-256 encryption, IP65 water resistance",
        "features": "Fingerprint unlock, keypad, remote access, auto-lock, tamper alarm."
    },
    "SmartThermostat X": {
        "specs": "7-inch LCD touchscreen, Z-Wave Plus, Energy Star certified, built-in humidity sensor",
        "features": "Learning algorithm, zone control, geofencing, energy usage reports."
    }
}

ORDER_DATABASE = {
    "AG12345": {"status": "Shipped", "tracking_id": "TRK987654", "estimated_delivery": "2026-03-10"},
    "AG67890": {"status": "Processing", "items": ["SmartBulb Pro", "SmartLock Elite"]},
    "AG11223": {"status": "Delivered", "delivery_date": "2026-02-28"},
    "AG44556": {"status": "Cancelled", "reason": "Customer request"}
}

TROUBLESHOOTING_GUIDES = {
    "SmartLock Elite": {
        "not connecting to wi-fi": "1. Ensure your Wi-Fi router is 2.4GHz. 2. Restart your router and the SmartLock. 3. Check for firmware updates in the app. 4. If issues persist, try a factory reset.",
        "battery draining fast": "1. Check for frequent unlocking/locking events. 2. Ensure firmware is up-to-date. 3. Reduce sensitivity of auto-lock feature. 4. Consider replacing with high-quality batteries."
    },
    "SmartBulb Pro": {
        "not responding": "1. Ensure the bulb is powered on. 2. Check Wi-Fi connection. 3. Try resetting the bulb by cycling power 5 times. 4. Re-pair with the app.",
        "flickering": "1. Ensure the bulb is compatible with your dimmer switch (if any). 2. Check for loose connections. 3. Try in a different fixture."
    }
}

# --- Define Custom Tools ---

@tool
def get_product_info(product_name: str) -> str:
    """Retrieves detailed information (specs, features) for a given product name.
    Use this tool when a user asks about product specifications, features, or details.
    Input should be the exact product name, e.g., 'SmartBulb Pro'.
    """
    product_name = product_name.strip()
    info = PRODUCT_DATABASE.get(product_name)
    if info:
        return f"Product: {product_name}\nSpecifications: {info['specs']}\nFeatures: {info['features']}"
    else:
        return f"Could not find information for product: {product_name}. Please check the product name."

@tool
def get_order_status(order_id: str) -> str:
    """Retrieves the current status and details for a given order ID.
    Use this tool when a user asks about their order's status, tracking, or delivery.
    Input should be the exact order ID, e.g., 'AG12345'.
    """
    order_id = order_id.strip()
    status_info = ORDER_DATABASE.get(order_id)
    if status_info:
        details = []
        for key, value in status_info.items():
            details.append(f"{key.replace('_', ' ').title()}: {value}")
        return f"Order ID: {order_id}\nStatus Details: {', '.join(details)}"
    else:
        return f"Could not find order with ID: {order_id}. Please verify the order ID."

@tool
def get_troubleshooting_guide(product_name: str, issue_description: str) -> str:
    """Provides troubleshooting steps for a specific product and issue.
    Use this tool when a user describes a problem with their device and needs help fixing it.
    Input should be the product name and a brief description of the issue, e.g., 'SmartLock Elite', 'not connecting to Wi-Fi'.
    """
    product_name = product_name.strip()
    issue_description = issue_description.strip().lower()

    product_issues = TROUBLESHOOTING_GUIDES.get(product_name)
    if product_issues:
        for issue_key, guide in product_issues.items():
            if issue_key in issue_description:
                return f"Troubleshooting steps for {product_name} - {issue_key}:\n{guide}"
        return f"No specific troubleshooting guide found for '{issue_description}' with {product_name}. Please describe the issue in more detail or try a different product."
    else:
        return f"No troubleshooting guides available for product: {product_name}."

# List of all tools available to the agent
agent_tools = [get_product_info, get_order_status, get_troubleshooting_guide]

# --- Initialize LLM ---
# Using gpt-4o for its advanced reasoning and function calling capabilities.
# Ensure OPENAI_API_KEY is set in your environment variables.
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# --- 1. Define the Agent Prompt ---
# The prompt guides the LLM's behavior and reasoning process.
# It includes a system message, chat history, and a placeholder for the agent's thought process (scratchpad).
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful and friendly customer support agent for AgenticGadgets Inc. Your goal is to assist customers with product information, order status, and troubleshooting. Use the provided tools to find accurate information. If a tool cannot provide the answer, politely inform the user and suggest alternative actions. Always maintain a helpful and professional tone."),
        MessagesPlaceholder("chat_history"), # Placeholder for conversational memory
        ("human", "{input}"),
        MessagesPlaceholder("agent_scratchpad"), # Placeholder for agent's thoughts and tool outputs
    ]
)

# --- 2. Create the Agent ---
# We use create_react_agent for its robust reasoning capabilities, allowing the LLM to decide when and how to use tools.
agent = create_react_agent(llm, agent_tools, prompt)

# --- 3. Set up Memory ---
# ConversationBufferMemory stores the chat history, which is then passed to the 'chat_history' placeholder in the prompt.
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# --- 4. Create Agent Executor ---
# The AgentExecutor is the runtime for the agent, managing the execution loop, tool calling, and memory.
agent_executor = AgentExecutor(
    agent=agent,
    tools=agent_tools,
    verbose=True, # Set to True to see the agent's thought process and tool usage
    memory=memory,
    handle_parsing_errors=True # Helps in gracefully handling cases where the LLM output isn't perfectly formatted
)

# --- 5. Test the Agent ---
print("\n--- AgenticGadgets Customer Support Agent Ready! ---\n")

def chat_with_agent(query: str):
    print(f"\nUser: {query}")
    response = agent_executor.invoke({"input": query})
    print(f"Agent: {response['output']}")

# Test Case 1: Product Information Query
chat_with_agent("What are the specifications of the SmartBulb Pro?")

# Test Case 2: Order Status Query
chat_with_agent("Can you tell me the status of my order AG12345?")

# Test Case 3: Troubleshooting Query
chat_with_agent("My SmartLock Elite isn't connecting to Wi-Fi. What should I do?")

# Test Case 4: Multi-turn conversation (using memory)
chat_with_agent("What about the SmartThermostat X?") # Follow-up to product info
chat_with_agent("And what if my SmartBulb Pro is flickering?") # New issue, but agent should remember context if relevant

# Test Case 5: Query that doesn't require a tool
chat_with_agent("Hello, how are you today?")

# Test Case 6: Query for a non-existent product
chat_with_agent("Tell me about the Quantum Router 5000.")

# Test Case 7: Query for a non-existent order
chat_with_agent("What's the status of order XYZ987?")

# Test Case 8: Complex query requiring multiple steps or clarification (agent should ask for more info or try best)
chat_with_agent("I have a problem with my SmartLock. It's not working.")

print("\n--- End of Agent Demonstration ---\n")
